## Index

- [About](#About)
    - [Context](#Context)
    - [Questions](#Questions)
    - [Data](#Data)
- [EDA](#EDA)
    - [Input](#Input)
    - [Relations](#Relations)
    - [Transform](#Transform)
- [Features](#Features)
- [Train](#Train)

## About
<div id="About"></div>

- Competition: [GoDaddy - Microbusiness Density Forecasting](https://www.kaggle.com/competitions/godaddy-microbusiness-density-forecasting)
- The Goal: Predict monthly microbusiness density in all counties from 2022-11 to 2023-06
- Metrics: SMAPE - percentage change


### Context
<div id="Context"></div>

Microbusiness (M.B.)
- a subcategory of a small business (fewer employees, less revenue and startup cost) 
- less than ten employees, 250k revenue and 50k startup cost
- example: home-based businesses, freelancers, consulting firms, photography studios, hair salons, drop shippers
- [economic profile](https://cdn.advocacy.sba.gov/wp-content/uploads/2021/08/30143723/Small-Business-Economic-Profile-US.pdf) show majority M.B. is in services (technical, professional..), construction, real estate, transportation, retail trade and health care 
- in the U.S., there are 31 million small businesses and 20k large businesses. 
- ~10-20% of employment is done by M.B.
- most M.B. are without employees ~90% (in small businesses, that is ~81%)
- loans common for M.B. [13k](https://www.sba.gov/funding-programs/loans/microloans) avg 


Trivia
- each year, ~1m establishments open and closes
- 2-year survival rate 60%
- small businesses contribution to [export value](https://cdn.advocacy.sba.gov/wp-content/uploads/2020/11/05122043/Small-Business-FAQ-2020.pdf) is low (32%)
- women own 40%, minorities 17%, immigrants 16% 
- 30% of the ones with employees are family owned (40% agricultural, 40% management)
- 25% home-based
- 5% franchises
- small businesses mostly incorporate and become S/C corporations, while 12% have a sole proprietorship. One person businesses 5% turn to S corp
- microfinance - Muhammad Yunus : [ted talk](https://www.youtube.com/watch?v=6UCuWxWiMaQ&ab_channel=TEDxTalks)


### Questions
<div id="Questions"></div>

- how does the host know about the number of employees and revenue to classify one site as M.B. (surveys, tax id, trademark ?) 
- any apparent anomalies in the change of density 
- how fast the data updates, any known lag 
- why there are incentives (loans, organizations, support) to create micro businesses (who benefits, how - how much). Do these incentives cause local/global density change 
- how do the competitors of the host (domain registrar) affect the values on a local scale (a county gets a discount, competitor loses clients, M.B. density is the same, but new clients move to the host)
- why are the top 10 highest the way they are (what makes some counties have more active M.B. or higher density)
- training period coincides with covid. Is there an increase in face-to-face business or more remote work, and more micro businesses? How could I find that out?
- why microbusinesses became small businesses? How often that happens? How often do they go out of business?

### Data
<div id="Data"></div>

- data comes from [venture forward](https://www.godaddy.com/ventureforward) sponsor of the competition.
- density values determined by registered domains on go daddy with active websites
- so the data is a filter of the M.B. population with website, who registerd using godaddy
- FIPS (federal information processing series) managed by ANSI. For example "49017" is the FIPS Code for Garfield County, Utah. "49" represents Utah and "017" represents Garfield County. (A county will always be assigned a single FIPS code but the county may contain dozens of ZIP Codes)


Datasets:
- U.S. Census Bureau (provided, population count lags 2 years and community survey)
- U.S. BUREAU OF LABOR STATISTICS (Employment data at bls.gov)


Mix resources #todo
- [GoDaddy Microbusiness Index July 2021](https://www.godaddy.com/ventureforward/wp-content/uploads/2021/07/GoDaddyUCLA_MicrobusinessIndex_July2021.pdf)
- [Best performing cities and microbusiness activity](https://www.godaddy.com/ventureforward/wp-content/uploads/2022/10/Best-Performing-Cities-and-Microbusiness-Activity.pdf)
- [gd2022 kaggleqrdl dataset](https://www.kaggle.com/datasets/kaggleqrdl/gd2022datasets/code)
- https://www.census.gov/programs-surveys/nonemployer-statistics/data/datasets.html
- https://data.sba.gov/dataset/
- https://www.census.gov/quickfacts/fact/table/US/PST045222
- https://www.census.gov/library/visualizations.html


## EDA
<div id="EDA"></div>

### Input
<div id="Input"></div>


In [ ]:
import numpy as np

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

import plotly.express as px
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
is_local_run = False

In [ ]:
if is_local_run:
    train = pd.read_csv('input/train.csv')
    test = pd.read_csv('input/test.csv')
    revealed = pd.read_csv('input/revealed_test.csv')
else:
    train = pd.read_csv('../input/godaddy-microbusiness-density-forecasting/train.csv')
    test = pd.read_csv('../input/godaddy-microbusiness-density-forecasting/test.csv')
    revealed = pd.read_csv('../input/godaddy-microbusiness-density-forecasting/revealed_test.csv')

train.shape, revealed.shape, test.shape, 

In [ ]:
train = pd.concat([train, revealed]).sort_values(by=['cfips','row_id'])
train = train.reset_index(drop=True)

In [ ]:
test = test.drop(test[test.first_day_of_month=='2022-11-01'].index)
test = test.drop(test[test.first_day_of_month=='2022-12-01'].index)
test = test.reset_index(drop=True)

In [ ]:
train.head()

- `cfips` -> unique id for county
- `active` -> number of microbusiness in a county
- `microbusiness_density` -> microbusiness per 100 people over the age 18 in that county

In [ ]:
# train

print(f"{train['cfips'].nunique()} : Number of Counties")
print(f"{train['state'].nunique()} : Number of States")
print(f"{train['first_day_of_month'].nunique()} : Months of train target data")

In [ ]:
# test

print(f"{test['first_day_of_month'].nunique()} : Months of test target data\n")
print(f"months to forcast: \n{test['first_day_of_month'].unique()}")

- `microbusiness_density` is the target mbdiable. 8 months of forcasting
- January 2023 public LB
- March, April and May 2023 is the Private LB

### Relations
<div id="Relations"></div>

In [ ]:
# how many months of observation for each fips (all 3135 have 39 obeservations)

train.groupby(['cfips'])['row_id'].count().value_counts()

In [ ]:
# total active vs date
# density mean vs date

from plotly.subplots import make_subplots
import plotly.graph_objects as go

mean_density_mb = train.groupby('first_day_of_month')['microbusiness_density'].mean().reset_index()
sum_active_mb = train.groupby('first_day_of_month')['active'].sum().reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Total Active","Density Mean"))

fig.add_trace(
    go.Scatter(
        x=sum_active_mb['first_day_of_month'],
        y=sum_active_mb['active'],
        name='active'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=mean_density_mb['first_day_of_month'], 
        y=mean_density_mb['microbusiness_density'],
        name='mean'),
    row=1, col=2
)
fig.show()

In [ ]:
from statsmodels.tsa.seasonal import DecomposeResult

def plot_seasonal_decompose(result: DecomposeResult, title="Seasonal Decomposition"):
    return (
        make_subplots(rows=4,cols=1,subplot_titles=["Observed", "Trend", "Seasonal", "Residuals"])
        .add_trace(go.Scatter(x=result.seasonal.index, y=result.observed, mode="lines"),row=1,col=1)
        .add_trace(go.Scatter(x=result.trend.index, y=result.trend, mode="lines"),row=2,col=1)
        .add_trace(go.Scatter(x=result.seasonal.index, y=result.seasonal, mode="lines"),row=3,col=1)
        .add_trace(go.Scatter(x=result.resid.index, y=result.resid, mode="lines"),row=4,col=1)
        .update_layout(height=800, title=title, margin=dict(t=50), title_x=0.5, showlegend=False)
    )

In [ ]:
import statsmodels.api as sm

mean_density_mb.set_index('first_day_of_month', inplace=True)
res = sm.tsa.seasonal_decompose(mean_density_mb['microbusiness_density'], period = 12, model = 'additive')
fig = plot_seasonal_decompose(result=res, title=f'Seasonal Decomposition of MBd')
fig.show()

- `microbusiness_density` has increasing
- shows yearly seasonal pattern

In [ ]:
df_std = train.groupby('cfips').std()
fig = px.histogram(
    df_std[df_std['microbusiness_density']<4],
    x='microbusiness_density',
    nbins=100,
    title='Density count')
fig.show()

In [ ]:
# num of counties per state

state_counties = train[['state','cfips']].drop_duplicates(subset=['cfips']).groupby(['state']).size().sort_values(ascending=False)

fig = px.bar(state_counties, title='Counties per state',
    labels={
        'value': 'num of counties'
    })
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# density vs date (of state x)

mbd_state = 'Arizona'
mbd_detail = train.loc[train.state==mbd_state]

fig = px.line(
    mbd_detail, 
    x="first_day_of_month", 
    y="microbusiness_density", 
    color='county', 
    title=f"Microbusiness density of {mbd_state} counties")
fig.show()

In [ ]:
# density vs state

fig = px.box(
    train,
    x="microbusiness_density",
    y="state",
    title=f"MB density distribution by states",
    height=1000)
fig.show()

In [ ]:
# county density mean

counties_mean = train.groupby('cfips').mean().reset_index()
counties_mean = counties_mean.merge(train[['cfips','county','state']].drop_duplicates(), on='cfips', how='inner')
counties_mean.head()

In [ ]:
# median and tail of density

qt = 0.97

fig = px.histogram(
    counties_mean, 
    x='microbusiness_density',     
    title=f"Median and Quantile {qt} of MB")

line_median = counties_mean['microbusiness_density'].median()
line_qt = counties_mean['microbusiness_density'].quantile(qt)

fig.add_vline(
    line_median,
    line_color='orange',
    line_dash='dash', 
    annotation_position="top right",
    annotation_text=f'm: {line_median:.1f}')

fig.add_vline(
    line_qt,
    line_color='green',
    line_dash='dash', 
    annotation_text=f'q : {line_qt:.1f}',
    annotation_position="top right")

fig.show()

In [ ]:
# Outliers: counties with a lot of microbusiness compared to population

outliers = counties_mean[counties_mean['microbusiness_density']>line_qt]
outliers.sort_values(by='microbusiness_density', ascending=False, inplace=True)
print(f'Outliers: {len(outliers)}\n')
outliers.head(3)

In [ ]:
fig = px.scatter(
    outliers, 
    x="active", 
    y="microbusiness_density", 
    hover_data=['state','county','cfips'],
    color='state',
    title='Outliers')

fig.update_layout(legend={'traceorder':'normal'})
fig.show()

In [ ]:
# sample cfip with sudden changes, selected from tableau

a_cfip = [12001, 31163, 25019, 55067, 8105, 16021, 6033, 32017, 8031, 31161, 46099, 46127, 56033, 10005, 17075, 38083, 24013, 56019, 51043, 2100, 30091, 19101, 32510, 20179, 46127, 41061]

In [ ]:
# density vs date (of state x)

fig = px.line(
    train.loc[train.cfips.isin(a_cfip)], 
    x="first_day_of_month", 
    y="microbusiness_density", 
    color='county', 
    title=f"Manuel sampling of counties")

fig.update_layout(legend={'traceorder':'normal'})
fig.show()

### Transform
<div id="Transform"></div>

In [ ]:
train.shape, test.shape

In [ ]:
# combine train test

train['istest'] = 0
test['istest'] = 1

ds = pd.concat((train, test)).sort_values(['cfips','row_id']).reset_index(drop=True)

In [ ]:
# date convert

ds['first_day_of_month'] = pd.to_datetime(ds["first_day_of_month"])
ds["year"] = ds["first_day_of_month"].dt.year
ds["month"] = ds["first_day_of_month"].dt.month

In [ ]:
# calculate population from active ms per 100 / density

ds['population']  = 100 * ds['active'] / ds['microbusiness_density']
ds['population'] = ds.groupby('cfips')['population'].fillna(method='ffill')
ds['population'] = ds['population'].round(0).astype(int)

In [ ]:
# fill test ds county and state from cfips grouping

ds['county'] = ds.groupby('cfips')['county'].fillna(method='ffill')
ds['state'] = ds.groupby('cfips')['state'].fillna(method='ffill')
ds.tail(2)

In [ ]:
ds.groupby(['cfips'])['row_id'].count().value_counts()

In [ ]:
# t -> time - county observation (train - test) (0-47)

ds['t'] = ds.groupby(['cfips'])['row_id'].cumcount()
ds.tail(2)

In [ ]:
# density vs date (of state x)

t_cfips = 56045
mbd_detail = ds.loc[ds.cfips == t_cfips]

fig = px.line(
    mbd_detail, 
    x="t", 
    y="microbusiness_density", 
    color='county', 
    title=f"Microbusiness density of cfips: {t_cfips}")
fig.show()

In [ ]:
# relative change according to lags
# from Giba's notebook

def calculate_lag_diff(lags):
    for lag in lags:
        ds[f'density_lag_{lag}'] = ds.groupby('cfips')['microbusiness_density'].shift(lag).bfill()
        ds[f'change_lag_{lag}'] = ds['microbusiness_density'] / ds[f'density_lag_{lag}'] 

        ds[f'diff_{lag}'] = (ds['microbusiness_density'] / ds[f'density_lag_{lag}']).fillna(1).clip(0, None) - 1
        ds.loc[(ds[f'density_lag_{lag}']==0), f'diff_{lag}'] = 0
        ds.loc[(ds[f'microbusiness_density']>0) & (ds[f'density_lag_{lag}'] ==0), f'diff_{lag}'] = 1
        ds[f'diff_{lag}'] = ds[f'diff_{lag}'].abs()

In [ ]:
lags = [1,3,5] # months

In [ ]:
calculate_lag_diff(lags)

fig = px.line(ds.groupby('t')[[f'diff_{lag}' for lag in lags]].sum(), title='lagging diff from avg')
fig.show()

In [ ]:
backup_ds = ds.copy()

In [ ]:
# identify a timestep of a cfip that has relative change more than threshold of the previous avg values

outliers = []
threshold = 0.23 # %23

for cfip in ds.cfips.unique():
    indices = (ds['cfips'] == cfip)
    county = ds.loc[indices].copy().reset_index(drop=True)
    mbd = county.microbusiness_density.values.copy()
    
    for i in range(39, 2, -1):
        limit = threshold * np.mean(mbd[:i])
        change = abs(mbd[i] - mbd[i-1])

        if (change >= limit):
            mbd[:i] *= (mbd[i] / mbd[i-1])
            outliers.append(cfip)

    mbd[0] = mbd[1]*0.99

    ds.loc[indices, 'microbusiness_density'] = mbd

outliers = np.unique(outliers)
print( f'num of outliers: {len(outliers)} in %{threshold}')

In [ ]:
# plot outliers 

fig = px.scatter(
    backup_ds.loc[backup_ds.cfips.isin(outliers)], 
    x="active", 
    y="microbusiness_density", 
    hover_data=['state','county','cfips'],
    color='state',
    title='Cfips with outlier time step')

fig.update_layout(legend={'traceorder':'normal'})
fig.show()

del backup_ds

In [ ]:
calculate_lag_diff(lags)

fig = px.line(
    ds.groupby('t')[[f'diff_{lag}' for lag in lags]].sum(),
    title='lagging diff from avg - after smoothing')
fig.show()

In [ ]:
# drop cols used for lag calculation

ds.drop([f'diff_{lag}' for lag in lags], axis=1, inplace=True)
ds.drop([f'change_lag_{lag}' for lag in lags], axis=1, inplace=True)
ds.drop([f'density_lag_{lag}' for lag in lags], axis=1, inplace=True)

In [ ]:
# SMAPE's target is the realtive change
# target becomes mb desnity change to next observation (~0-1)

ds['target'] = ds.groupby('cfips')['microbusiness_density'].shift(-1)
ds['target'] = ds['target']/ds['microbusiness_density'] - 1

In [ ]:
ds.head()

In [ ]:
fig = px.histogram(
    ds['target'].clip(-0.4, 0.4),
    nbins=200,
    title='density % cahnge each month')
fig.show()

In [ ]:
ds['lastactive'] = ds.groupby('cfips')['active'].transform('last')

dtarget = ds.loc[ds.t==28].groupby('cfips')['microbusiness_density'].agg('last')
ds['lasttarget'] = ds['cfips'].map(dtarget)

In [ ]:
# Last known density and active of each cfip

fig = make_subplots(rows=1, cols=2, subplot_titles=("Last Active","Last Target"))


fig.add_trace(
    go.Histogram(x=ds['lastactive'].clip(0, 6000), name='active'),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=ds['lasttarget'].clip(0, 20), name='target'),
    row=1, col=2
)

fig.update_layout(
    title_text='Last known density and active of each cfip',
    xaxis_title_text='value',
    yaxis_title_text='count',
    bargap=0.2,
    bargroupgap=0.1
)

fig.show()

### Stats models
<div id="Stats models"></div>

In [ ]:
from statsmodels.tsa.seasonal import STL

for cfip in np.random.choice(train['cfips'].unique(), 2):
    temp_ds = ds.loc[(ds['cfips'] == cfip) & (ds['istest'] == False), ['microbusiness_density', 'first_day_of_month']]
    
    temp_series = pd.Series(
        temp_ds['microbusiness_density'].values.squeeze(),
        index = temp_ds['first_day_of_month'],
        name = f'cfips{cfip}')

    stl = STL(temp_series, seasonal = 13)
    res = stl.fit()

    fig = plot_seasonal_decompose(result=res, title=f'Seasonal Decomposition of cfip: {cfip}')
    fig.show()

## Features
<div id="Features"></div>

In [ ]:
if is_local_run:
    county = pd.read_csv('input/us-county-areas.csv', dtype={'statefp': str, 'countyfp': str})
    census = pd.read_csv('input/census_starter.csv')
    employment = pd.read_csv('input/employment.csv')
else:
    county = pd.read_csv('/kaggle/input/us-county-areas//us-county-areas.csv', dtype={'statefp': str, 'countyfp': str})
    census = pd.read_csv('/kaggle/input/godaddy-microbusiness-density-forecasting/census_starter.csv')
    employment = pd.read_csv('/kaggle/input/employment-godaddy/employment.csv')

In [ ]:
# add census data

ds = ds.merge(census,on="cfips", how="left", sort=False)

In [ ]:
# add county area data

county['cfips'] = (county['statefp'] + county['countyfp']).astype(int)

ds = ds.merge(county[["cfips","area"]],on="cfips", how="left", sort=False)
ds['population_density'] = ds['population']/ds['area']

In [ ]:
# add employement data
employment['first_day_of_month'] = pd.to_datetime(employment["first_day_of_month"])

ds = ds.merge(employment[["cfips","first_day_of_month",'emp']],on=['cfips','first_day_of_month'], how="left", sort=False)

In [ ]:
# factorize state/county

ds['county_i'] = (ds['county'] + ds['state']).factorize()[0]
ds['state_i'] = ds['state'].factorize()[0]

In [ ]:
def build_window_features(ds, target, target_act, lags):
    feats = []
    for lag in range(1, lags):
        ds[f'micbd{lag}'] = ds.groupby('cfips')[target].shift(lag)
        ds[f'active_lag_{lag}'] = ds.groupby('cfips')[target_act].diff(lag)
        feats.append(f'micbd{lag}')
        feats.append(f'active_lag_{lag}')
        
    lag = 1
    for window in [2, 4, 6, 8, 10]:
        ds[f'mbd_rollmea{window}_{lag}'] = ds.groupby('cfips')[f'micbd{lag}'].transform(lambda s: s.rolling(window, min_periods=1).sum())        
        feats.append(f'mbd_rollmea{window}_{lag}')
        
    return ds, feats

In [ ]:
def build_census_features(ds):    
        
    ds['pct_bb_diff'] = ds['pct_bb_2021'] -ds['pct_bb_2017']
    ds['pct_college_diff'] = ds['pct_college_2021'] -ds['pct_college_2017']
    ds['pct_foreign_born_diff'] = ds['pct_foreign_born_2021'] -ds['pct_foreign_born_2017']    
    ds['pct_it_workers_diff'] = ds['pct_it_workers_2021'] -ds['pct_it_workers_2017']
    ds['median_hh_inc_diff'] = ds['median_hh_inc_2021'] -ds['median_hh_inc_2017']

    census_features  = ['pct_bb_2021','pct_college_2021','pct_foreign_born_2021',
                        'pct_it_workers_2021','median_hh_inc_2021', 'pct_bb_diff',
                        'pct_college_diff','pct_foreign_born_diff','pct_it_workers_diff',
                        'median_hh_inc_diff']
    return ds, census_features    

In [ ]:
# Build Features based in lag of target
features = ['state_i', 'county_i']
ds, feats = build_window_features(ds, 'target', 'active', lags = 5)
features += feats

# census:
ds, census_features = build_census_features(ds)
features += census_features

# county pop:
features += ['population_density']

print(features)
ds.loc[ds.t==38, features].head(10)

In [ ]:
fig = px.histogram(
    ds['lasttarget'].clip(0,10),
    nbins=100)
fig.show()

## Train
<div id="Train"></div>

In [ ]:
def smape(y_true, y_pred):
    smap = np.zeros(len(y_true))
    
    num = np.abs(y_true - y_pred)
    dem = ((np.abs(y_true) + np.abs(y_pred)) / 2)
    
    pos_ind = (y_true!=0)|(y_pred!=0)
    smap[pos_ind] = num[pos_ind] / dem[pos_ind]
    
    return 100 * np.mean(smap)

def vsmape(y_true, y_pred):
    smap = np.zeros(len(y_true))
    
    num = np.abs(y_true - y_pred)
    dem = ((np.abs(y_true) + np.abs(y_pred)) / 2)
    
    pos_ind = (y_true!=0)|(y_pred!=0)
    smap[pos_ind] = num[pos_ind] / dem[pos_ind]
    
    return 100 * smap

In [ ]:
# prep
# ds.loc[(ds.target.isna()) & (ds.istest == 0)]


ds.loc[ds['cfips']==28055, 'target'] = 0.0
ds.loc[ds['cfips']==48269, 'target'] = 0.0

ds[f'mbd_lag_1'] = ds.groupby('cfips')['microbusiness_density'].shift(1).bfill()

ds['ypred_last'] = np.nan
ds['ypred'] = np.nan
ds['k'] = 1.

In [ ]:
# petersorensen360

import xgboost as xgb

best_rounds = []

# settings
month_lookahead = 1
last_active_threshold = 1.75
last_target_threshold = 1.00
last_t_train = 40 + 1 - month_lookahead

for ctime in range(29, last_t_train):
    print('current t:', ctime)
    
    # gpu_hist - white

    model = xgb.XGBRegressor(
        objective='reg:pseudohubererror',
        tree_method='hist',
        n_estimators=4999,
        learning_rate=0.007,
        max_leaves = 17,
        subsample=0.50,
        colsample_bytree=0.50,
        max_bin=4096,
        n_jobs=-1,
        eval_metric='mae',
        early_stopping_rounds=70,
    )
            
    train_indices = (ds.istest==0) & (ds.t  < ctime) & (ds.t >= 1) & (ds.lastactive>last_active_threshold)  & (ds.lasttarget>last_target_threshold) 
    ct_indices = (ds.istest==0) & (ds.t == ctime)
    
    model.fit(
        ds.loc[train_indices, features],
        ds.loc[train_indices, 'target'].clip(-0.0043, 0.0043), #todo
        eval_set=[(ds.loc[ct_indices, features], ds.loc[ct_indices, 'target'])],
        verbose=500,
    )

    best_rounds.append(model.best_iteration)
    ypred = model.predict(ds.loc[ct_indices, features])
    
    ds.loc[ct_indices, 'k'] = ypred + 1
    ds.loc[ct_indices,'k'] = ds.loc[ct_indices,'k'] * ds.loc[ct_indices,'microbusiness_density']

    # Validate
    lastval = ds.loc[ds.t==ctime, ['cfips', 'microbusiness_density']].set_index('cfips').to_dict()['microbusiness_density']
    dt = ds.loc[ds.t==ctime, ['cfips', 'k']].set_index('cfips').to_dict()['k']
    

    df = ds.loc[ds.t==(ctime + month_lookahead), ['cfips', 'microbusiness_density', 'state', 'lastactive', 'mbd_lag_1']].reset_index(drop=True)
    df['pred'] = df['cfips'].map(dt)
    df['lastval'] = df['cfips'].map(lastval)
    
    # skip predictions on the ones outside defined thresholds, assign last value
    df.loc[df['lastactive']<=last_active_threshold, 'pred'] = df.loc[df['lastactive']<=last_active_threshold, 'lastval']
    df.loc[df['lastval']<=last_target_threshold, 'pred'] = df.loc[df['lastval']<=last_target_threshold, 'lastval']

    # assign predictions
    ds.loc[ds.t==(ctime+month_lookahead), 'ypred'] = df['pred'].values
    ds.loc[ds.t==(ctime+month_lookahead), 'ypred_last'] = df['lastval'].values

    print(f'ctime: {ctime}')
    print('Last Value SMAPE:', smape(df['microbusiness_density'], df['lastval']) )
    print('XGB SMAPE:', smape(df['microbusiness_density'], df['pred']))
    print()

In [ ]:
print(ds.loc[ds.t>=29,["t",'microbusiness_density',"ypred","ypred_last"]].head(20))

In [ ]:
ind = (ds.t>=29+month_lookahead)&(ds.t<=last_t_train+month_lookahead-1)
print('Last Value SMAPE:', smape( ds.loc[ind, 'microbusiness_density'],  ds.loc[ind, 'ypred_last'] ) )
print('XGB SMAPE:', smape( ds.loc[ind, 'microbusiness_density'],  ds.loc[ind, 'ypred'] ) )

In [ ]:
ds['error'] = vsmape(ds['microbusiness_density'], ds['ypred'])
ds['error_last'] = vsmape(ds['microbusiness_density'], ds['ypred_last'])
ds.loc[(ds.t==30+month_lookahead-1), ['microbusiness_density', 'ypred', 'error', 'error_last'] ]

In [ ]:
dt = ds.loc[(ds.t>=30+month_lookahead-1) & (ds.t<=last_t_train-1+month_lookahead) ]
dt['hit'] = dt['error'] - dt['error_last']
dt = dt[dt['hit']>0]

fig = px.histogram(dt['microbusiness_density'].clip(0,20), nbins=100)
fig.show()

In [ ]:
dt = ds.loc[(ds.t>=30+month_lookahead-1) & (ds.t<=last_t_train) ].groupby('state')['error', 'error_last'].mean()
dt['hit'] = dt['error'] - dt['error_last']

blacklist_states = dt.loc[dt['hit']>0].index.values.tolist()
print(f"blacklist states:{blacklist_states}")

In [ ]:
dt = ds.loc[(ds.t>=30+month_lookahead-1) & (ds.t<=last_t_train)].groupby(['cfips','t'])['error', 'error_last','state'].last()

In [ ]:
cfips_blacklist_criteria = 0.6

dt = dt[~dt['state'].isin(blacklist_states)]
dt['miss'] = dt['error'] > dt['error_last']
dt = dt.groupby('cfips')['miss'].mean()

print("Criteria {} chosen, gives {} cfips to blacklist:".format(cfips_blacklist_criteria,len(dt.loc[dt>=cfips_blacklist_criteria].index)))
print()
blacklist_cfips = dt.loc[dt>=cfips_blacklist_criteria].index.values.tolist()
print(f'num of county blacklisted: {len(blacklist_cfips)}')

In [ ]:
import random

for fips in random.sample(list(dt.index), 3):
    px.line(
        ds.loc[ds.cfips == fips],
        x='t',
        y=['microbusiness_density', 'ypred', 'ypred_last'],
        title=f'cfips: {fips}').show()
    

In [ ]:
np.mean(best_rounds), np.median(best_rounds), best_rounds

In [ ]:
best_rounds = int(np.median(best_rounds)+1)
best_rounds

In [ ]:
ctime = last_t_train
print(ctime)
ds['k'] = 1.
model0 = xgb.XGBRegressor(
    objective='reg:pseudohubererror',
    tree_method="hist", # gpu_hist 
    n_estimators=best_rounds,
    learning_rate=0.007,
    max_leaves = 31,
    subsample=0.60,
    colsample_bytree=0.50,
    max_bin=4096,
    n_jobs=-1,
    eval_metric='mae',
)
model1 = xgb.XGBRegressor(
    objective='reg:pseudohubererror',
    tree_method="hist",
    n_estimators=best_rounds,
    learning_rate=0.007,
    max_leaves = 31,
    subsample=0.60,
    colsample_bytree=0.50,
    max_bin=4096,
    n_jobs=-1,
    eval_metric='mae',
)

train_indices = (ds.istest==0) & (ds.t  < ctime) & (ds.t >= 1) & (ds.lastactive>last_active_threshold)  & (ds.lasttarget>last_target_threshold) 
valid_indices = (ds.t == ctime)
model0.fit(
    ds.loc[train_indices, features],
    ds.loc[train_indices, 'target'].clip(-0.0044, 0.0045),
)
model1.fit(
    ds.loc[train_indices, features],
    ds.loc[train_indices, 'target'].clip(-0.0044, 0.0045),
)

ypred = (model0.predict(ds.loc[valid_indices, features]) + model1.predict(ds.loc[valid_indices, features]))/2
ds.loc[valid_indices, 'k'] = ypred + 1.
ds.loc[valid_indices,'k'] = ds.loc[valid_indices,'k'] * ds.loc[valid_indices,'microbusiness_density']

# Validate
lastval = ds.loc[ds.t==ctime, ['cfips', 'microbusiness_density']].set_index('cfips').to_dict()['microbusiness_density']
dt = ds.loc[ds.t==ctime, ['cfips', 'k']].set_index('cfips').to_dict()['k']

In [ ]:
print(ds.loc[valid_indices,['t','k']].tail(10))

In [ ]:
model_ds = pd.DataFrame({'names': model0.feature_names_in_, 'importance':model0.feature_importances_})
model_ds = model_ds.sort_values(by='importance', ascending=False)
model_ds.reset_index(drop=True)

fig = px.bar(model_ds, y='names', x='importance', orientation='h')
fig.show()

In [ ]:
# Feature correlation

fig = px.imshow(ds.loc[train_indices, sorted(features)].corr(), text_auto=True,  aspect="auto")
fig.show()

In [ ]:
import shap
explainer = shap.TreeExplainer(model0)
shap_values = explainer.shap_values(ds.loc[train_indices, features])
shap.summary_plot(shap_values, ds.loc[train_indices, features], plot_type="bar")
shap.summary_plot(shap_values, ds.loc[train_indices, features])

In [ ]:
df = ds.loc[ds.t==(ctime + month_lookahead), ['cfips', 'microbusiness_density', 'state', 'lastactive', 'mbd_lag_1']].reset_index(drop=True)
df['pred'] = df['cfips'].map(dt)
df['lastval'] = df['cfips'].map(lastval)

df.loc[df['lastactive'] <= last_active_threshold, 'pred'] = df.loc[df['lastactive'] <= last_active_threshold, 'lastval']
df.loc[df['lastval'] <= last_target_threshold, 'pred'] = df.loc[df['lastval'] <= last_target_threshold, 'lastval']
df.loc[df['state'].isin(blacklist_states), 'pred'] = df.loc[df['state'].isin(blacklist_states), 'lastval']
df.loc[df['cfips'].isin(blacklist_cfips), 'pred'] = df.loc[df['cfips'].isin(blacklist_cfips), 'lastval']
ds.loc[ds.t == (ctime+month_lookahead), 'ypred'] = df['pred'].values
ds.loc[ds.t == (ctime+month_lookahead), 'ypred_last'] = df['lastval'].values

In [ ]:
ds[['cfips','microbusiness_density','t','ypred','ypred_last','k']].tail(10)

In [ ]:
print(ds.loc[ds.t==39,['cfips','microbusiness_density','t','ypred','ypred_last','k']].tail(10))

In [ ]:
if is_local_run:
    adult_2020_2021 = pd.read_csv('input/adult_2020_2021.csv')
else:
    adult_2020_2021 = pd.read_csv('../input/employment-godaddy/adult_2020_2021.csv')

In [ ]:
adult_2020_2021.head()

In [ ]:
ds = ds.merge(adult_2020_2021,on="cfips", how="left", sort=False)

In [ ]:
ds.loc[ds['cfips']==28055, 'microbusiness_density'] = 0
ds.loc[ds['cfips']==48269, 'microbusiness_density'] = 1.73


print("Month predicted",last_t_train+month_lookahead)

dt = ds.loc[ds.t==last_t_train+month_lookahead, ['cfips', 'ypred']].set_index('cfips').to_dict()['ypred']


In [ ]:
ds.loc[ds.first_day_of_month=='2022-11-01','istest'] = 1
ds.loc[ds.first_day_of_month=='2022-12-01','istest'] = 1

In [ ]:
test = ds.loc[ds.istest==1, ['row_id', 'cfips','microbusiness_density', 'adult2020', 'adult2021']].copy()
test['microbusiness_density'] = test['cfips'].map(dt)

In [ ]:
test.head()

In [ ]:
test.microbusiness_density = test.microbusiness_density * test.adult2020 / test.adult2021
test.head()

In [ ]:
test[['row_id','microbusiness_density']].to_csv('submission.csv', index=False)